# Phase 2c — Causal activation patch
Does the up-block **self-attention** site *cause* the count? For each seed we capture the attn1 activations from a **donor** prompt (e.g. 'five cats') and inject them into a **source** run ('two cats') at the commitment steps, then check whether the OUTPUT count moves toward the donor.

Both directions (2->inject5 should raise; 5->inject2 should lower). Effect in both directions = causal; ~0 = correlational (like prior steering).

**Runtime:** GPU.

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/serinaqin/T2I-Count-Anomaly.git
    %cd T2I-Count-Anomaly
!pip install -q -r requirements.txt
!pip install -q pytest groundingdino-py

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, os, yaml
from src.prompts import build_prompt
from src.pipeline import (load_sdxl, generate, catalog_attention_sites,
                          select_probe_sites, generate_and_capture,
                          raw_reducer, generate_with_patch)
from src.detector import Detector
from src.scoring import count_from_detections
from src.config import load_config

In [ ]:
cfg = load_config('configs/phase2c.yaml')
raw = yaml.safe_load(open('configs/phase2c.yaml'))
pairs, patch_steps, block = raw['pairs'], raw['patch_steps'], raw['patch_block']
obj = cfg.objects[0]
print('pairs', pairs, '| patch steps', patch_steps, '| block', block, '| obj', obj)

In [ ]:
pipe = load_sdxl()
det = Detector()
sites = [s for s in select_probe_sites(catalog_attention_sites(pipe.unet))
         if block in s and s.endswith('attn1')]
print(len(sites), 'patch sites:', sites)
def cnt(img):
    return count_from_detections(det.detect(img, [obj]), obj, cfg.score_threshold)

In [ ]:
# For each (source, donor) pair and seed: donor-capture, baseline, patched.
rows, gallery = [], []
for src, dnr in pairs:
    sp, dp = build_prompt(src, obj), build_prompt(dnr, obj)
    for seed in cfg.seeds:
        img_d, snaps = generate_and_capture(pipe, dp, seed, sites, patch_steps,
                                            cfg.num_inference_steps, reducer=raw_reducer)
        img_b = generate(pipe, sp, seed, cfg.num_inference_steps)
        img_p = generate_with_patch(pipe, sp, seed, snaps, cfg.num_inference_steps)
        cd, cb, cp = cnt(img_d), cnt(img_b), cnt(img_p)
        direction = 'up' if src < dnr else 'down'
        rows.append({'src': src, 'dnr': dnr, 'seed': seed, 'direction': direction,
                     'c_donor': cd, 'c_base': cb, 'c_patch': cp})
        if direction == 'up':
            gallery.append((seed, img_b, img_p, img_d, cb, cp, cd))
df = pd.DataFrame(rows)
df['delta'] = df['c_patch'] - df['c_base']
os.makedirs('results', exist_ok=True)
df.to_csv('results/phase2c_patch.csv', index=False)
df

In [ ]:
# Effect per direction: does patching move the count the expected way?
for direction, g in df.groupby('direction'):
    md = g['delta'].mean()
    frac = (g['delta'] > 0).mean() if direction == 'up' else (g['delta'] < 0).mean()
    print(f'{direction:>4}: mean(patched-baseline) = {md:+.2f} | '
          f'baseline={g.c_base.mean():.2f} patched={g.c_patch.mean():.2f} '
          f'donor={g.c_donor.mean():.2f} | frac moved expected way = {frac:.2f}')

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
col = {'up': 'C0', 'down': 'C3'}
for direction, g in df.groupby('direction'):
    for _, r in g.iterrows():
        axes[0].plot([0, 1], [r.c_base, r.c_patch], color=col[direction],
                     alpha=0.5, marker='o')
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['baseline', 'patched'])
axes[0].set_ylabel('output count')
axes[0].set_title('Per-seed: does injecting the donor move the count?')
means = df.groupby('direction')['delta'].mean()
axes[1].bar(means.index, means.values, color=[col[d] for d in means.index])
axes[1].axhline(0, color='k', lw=0.8)
axes[1].set_ylabel('mean (patched - baseline) count')
axes[1].set_title('Causal effect (up should be +, down should be -)')
plt.tight_layout()
plt.savefig('results/phase2c_effect.png', dpi=100, bbox_inches='tight'); plt.show()

In [ ]:
# Eyeball: baseline | patched | donor for a few seeds (up direction).
show = gallery[:3]
fig, axes = plt.subplots(len(show), 3, figsize=(9, 3 * len(show)))
if len(show) == 1: axes = axes[None, :]
for row, (seed, ib, ip, idn, cb, cp, cd) in enumerate(show):
    for col_i, (im, ti) in enumerate([(ib, f'baseline={cb}'), (ip, f'patched={cp}'),
                                      (idn, f'donor={cd}')]):
        axes[row, col_i].imshow(im); axes[row, col_i].axis('off')
        axes[row, col_i].set_title(f'seed {seed} | {ti}', fontsize=9)
plt.tight_layout()
plt.savefig('results/phase2c_eyeball.png', dpi=90, bbox_inches='tight'); plt.show()

## How to read this
- **up mean delta clearly > 0 AND down mean delta clearly < 0** = injecting the donor's up-block self-attention *moves the output count toward the donor* -> this site **causally controls** the count. That's the realization mechanism; Phase 4 mitigation intervenes here.
- **delta ~ 0 (or only one direction works)** = correlational, not causal (the same result as the project's earlier steering). The count is set elsewhere -> widen the patch (more steps/sites), move to a different block, or upstream toward the initial noise.
- **Eyeball:** in the up rows, does the patched image actually show more animals than the baseline (toward the donor)? Counts only mean something if the images visibly change.